In [9]:
!pip install scikit-image

Defaulting to user installation because normal site-packages is not writeable
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.9 MB 1.6 MB/s eta 0:00:08
   --- ------------------------------------ 1.0/11.9 MB 2.0 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/11.9 MB 2.5 MB/s eta 0:00:04
   -------- ------------------------------- 2.6/11.9 MB 2.8 MB/s eta 0:00:04
   --------------- ------------------------ 4.7/11.9 MB 4.2 MB/s eta 0:00:02
   ------------------------ --------------- 7.3/11.9 MB 5.5 MB/s eta 0:00:01
   -------------------------------- ------- 9.7/11.9 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------  11.8/11.9 MB 6.9 MB/s eta 0:00:01
   ---------------------------------------- 11.9/11.9 MB 6.7 MB/s eta 0:00:00
Using cached networkx-

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import cv2, numpy, skimage, matplotlib, PIL
print("All good ✅")

All good ✅


In [14]:
# ── Install dependencies if needed ──────────────────────────────────────────
# Run this cell once, then restart kernel
# !pip install opencv-python scikit-image matplotlib pillow numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import sys

# ── Point this to wherever your preprocessing.py lives ──────────────────────

from preprocessor1 import (
    correct_blue_shift,
    extract_coral_roi,
    extract_glcm_features,
    generate_rcbi_heatmap,
)

# ── CHANGE THIS to any image you want to test ────────────────────────────────
IMAGE_PATH = "dataset/healthy/check1.jpg"


# ── Helper ───────────────────────────────────────────────────────────────────
def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# ── Run all stages ───────────────────────────────────────────────────────────
img_bgr    = cv2.imread(IMAGE_PATH)
assert img_bgr is not None, f"Could not load image at: {IMAGE_PATH}"

corrected        = correct_blue_shift(img_bgr)
mask, masked     = extract_coral_roi(corrected)
glcm             = extract_glcm_features(masked, mask)
rcbi_map, heatmap = generate_rcbi_heatmap(corrected, mask)

coral_pixels = rcbi_map[mask > 0]
rcbi_mean    = float(coral_pixels.mean()) if len(coral_pixels) > 0 else 0.0
coverage     = (mask > 0).sum() / mask.size


# ── Display ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f"DIP Preprocessing — {IMAGE_PATH.split('/')[-1]}", fontsize=13)

panels = [
    (bgr_to_rgb(img_bgr),    "Original (raw)"),
    (bgr_to_rgb(corrected),  "Stage 1: Blue-shift corrected"),
    (mask,                   "Stage 2: ROI mask (binary)"),
    (bgr_to_rgb(masked),     "Stage 2: Masked image"),
    (bgr_to_rgb(heatmap),    f"Stage 4: rCBI heatmap  (mean={rcbi_mean:.3f})"),
    (rcbi_map,               "Stage 4: Raw rCBI values"),
]

for ax, (img, title) in zip(axes.flat, panels):
    if img.ndim == 2:
        if img.dtype == np.float32 or img.max() <= 1.0:
            im = ax.imshow(img, cmap="RdYlBu_r", vmin=-0.5, vmax=0.5)
            plt.colorbar(im, ax=ax, fraction=0.046)
        else:
            ax.imshow(img, cmap="gray")
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()


# ── Print feature values ─────────────────────────────────────────────────────
print("\nGLCM Texture Features:")
for k, v in glcm.items():
    print(f"  {k}: {v:.4f}")
print(f"\nrCBI mean (coral region): {rcbi_mean:.4f}")
print(f"Coral coverage:           {coverage:.1%}")


# ── Optional: view with cv2.imshow ───────────────────────────────────────────
# Uncomment these lines if you want the OpenCV popup window
# Note: cv2.imshow may not work in all Jupyter environments (works in VS Code)

# cv2.imshow("Original",        img_bgr)
# cv2.imshow("Blue corrected",  corrected)
# cv2.imshow("ROI mask",        mask)
# cv2.imshow("Masked image",    masked)
# cv2.imshow("rCBI heatmap",    heatmap)
# cv2.waitKey(0)
# cv2.destroyAllWindows()


GLCM Texture Features:
  glcm_contrast: 13.9974
  glcm_correlation: 0.9373
  glcm_energy: 0.7834
  glcm_homogeneity: 0.8269

rCBI mean (coral region): 0.4679
Coral coverage:           21.1%


C:\Users\LEGION\AppData\Local\Temp\ipykernel_10808\3953806748.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
import os

print("Current working dir:", os.getcwd())
print("Files here:", os.listdir())

Current working dir: D:\CoralReef
Files here: ['.ipynb_checkpoints', 'dataset', 'preprocessor1.ipynb', 'test1.ipynb']
